# Scenario: Customer Support Chatbot Workflow
 Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

 State Definition (BotState)
 - The chatbot keeps track of:
 - The question asked by the customer.
 - The answer generated.
 - The history of all past questions.
 - Think of this as the chatbot’s notebook where it records the conversation.

 Nodes (Functions)
 - get_answer:
 When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
 It also adds the question to the history log.
 - format_output:
 Before sending the response back to the customer, the chatbot reformats it into a friendly style:
 “Bot says: Answer to: What are your store hours?”

 Graph Flow
 - The chatbot starts at the get_answer node (entry point).
 - Once the answer is generated, it flows to the format_output node.
 - Finally, the conversation ends at END, meaning the chatbot has
  delivered its response.

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}"
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

# Scenario: Customer Support Chatbot (Question-Based)
 Imagine a company has deployed a chatbot that answers customer
  questions by calling the Groq API. The workflow is modeled as a
  graph of states, where each customer query flows through nodes until
   a response is delivered.

1. State Definition
 The chatbot maintains a notebook-like state:
 - question → The customer’s query.
 - answer → The response generated by Groq.
 - history → A log of all past questions.

In [33]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import requests
from google.colab import userdata


# =========================
# 1. STATE DEFINITION
# =========================
class BotState(TypedDict):
    question: str
    answer: str
    history: List[str]

# =========================
# 2. LLM NODE (Groq API)
# =========================
def llm_node(state: BotState):
    question = state["question"]
    history = state["history"]

    # ✅ Fetch API key safely
    groq_api_key = userdata.get("NewApi")

    if not groq_api_key:
        raise ValueError("❌ Groq API key not found")

    # ✅ Convert history into messages
    messages = [{"role": "system", "content": "You are a helpful customer support assistant."}]

    for q in history:
        messages.append({"role": "user", "content": q})

    messages.append({"role": "user", "content": question})

    # ✅ Call Groq API
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {groq_api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": messages,
            "temperature": 0.7
        },
        timeout=15
    )

    print("Status:", response.status_code)
    print("Response:", response.text)

    if response.status_code != 200:
        raise Exception(f"❌ API Error: {response.text}")

    data = response.json()
    answer = data["choices"][0]["message"]["content"]

    return {
        "answer": answer,
        "history": history + [question]
    }

# =========================
# 3. FORMAT NODE
# =========================
def format_node(state: BotState):
    return {
        "answer": f"🤖 Support Bot: {state['answer']}",
        "history": state["history"]
    }

# =========================
# 4. BUILD GRAPH
# =========================
graph = StateGraph(BotState)

graph.add_node("llm", llm_node)
graph.add_node("format", format_node)

graph.set_entry_point("llm")

graph.add_edge("llm", "format")
graph.add_edge("format", END)

app = graph.compile()

# =========================
# 5. MULTI-TURN CHAT LOOP
# =========================
if __name__ == "__main__":
    state = {
        "question": "",
        "answer": "",
        "history": []
    }

    print("💬 Customer Support Chatbot (type 'exit' to stop)\n")

    while True:
        user_input = input("👤 You: ")

        if user_input.lower() == "exit":
            break

        state["question"] = user_input

        result = app.invoke(state)

        print(result["answer"])

        # ✅ Update state for memory
        state["history"] = result["history"]

💬 Customer Support Chatbot (type 'exit' to stop)

👤 You: what is ml
Status: 200
Response: {"id":"chatcmpl-df22d7f1-096c-4d6f-a10f-cf3f5fcbfa0c","object":"chat.completion","created":1773986858,"model":"llama-3.1-8b-instant","choices":[{"index":0,"message":{"role":"assistant","content":"ML stands for Machine Learning. It's a subset of Artificial Intelligence (AI) that focuses on developing algorithms and statistical models that enable computers to learn from data, make predictions, and improve their performance over time.\n\nMachine Learning involves training a model on a dataset, which allows the model to identify patterns and relationships within the data. This enables the model to make accurate predictions or take actions based on new, unseen data.\n\nThere are several types of Machine Learning:\n\n1. **Supervised Learning**: The model is trained on labeled data, where the correct output is already known. The goal is to learn a mapping between inputs and outputs.\n2. **Unsupervised Le

#Scenario: AI-Powered Study Assistant (Flashcard-Based)
1. State Definition
The assistant maintains a notebook-like state for each learner:
- topic → The subject the learner is studying (e.g., "Photosynthesis").
- flashcard → A generated Q&A card created by Groq (question on one side, answer on the other).
- progress → A log of all past flashcards attempted, including whether the learner got them correct or not.

2. Workflow (Graph of States)
Each learner interaction flows through nodes until a flashcard is delivered:
- Input Node
- Learner provides a topic or asks for practice (e.g., "Test me on cell biology").
- State updates: topic = "cell biology"
- Generation Node (Groq API)
- Groq generates a flashcard:
- flashcard.question = "What organelle is known as the powerhouse of the cell?"
- flashcard.answer = "Mitochondria"
- Response Node
- Assistant presents the flashcard question to the learner.
- Evaluation Node
- Learner responds with their answer.
- Assistant checks correctness and updates progress.
- History Node
- Logs the flashcard attempt:
- progress = [{question, learner_answer, correct/incorrect}]

In [6]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
import re
from google.colab import userdata

userdata.get('NewApi')

def get_api_key():
    return userdata.get('NewApi')

# =========================
# 1. STATE
# =========================
class BotState(TypedDict):
    topic: str
    flashcard: Dict
    learner_answer: str
    is_correct: bool
    progress: List[Dict]
    exit: bool   # ✅ NEW

# =========================
# 2. INPUT NODE
# =========================
def input_node(state: BotState):
    topic = input("\n📘 Enter topic (or 'exit'): ")

    if topic.lower() == "exit":
        print("👋 Exiting Study Assistant...")
        return {"exit": True}

    return {"topic": topic, "exit": False}

def route_after_input(state: BotState):
    if state.get("exit"):
        return END
    return "generate"

# =========================
# 3. GENERATE FLASHCARD (LLM)
# =========================
def generate_flashcard(state: BotState):
    api_key = get_api_key()
    topic = state["topic"]

    prompt = f"""
    Generate ONE flashcard for studying {topic}.

    Format STRICTLY:
    Question: <question>
    Answer: <answer>
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.7
        },
        timeout=15
    )

    if response.status_code != 200:
        raise Exception(response.text)

    content = response.json()["choices"][0]["message"]["content"]

    # ✅ Robust parsing
    q_match = re.search(r"Question:\s*(.*)", content, re.IGNORECASE)
    a_match = re.search(r"Answer:\s*(.*)", content, re.IGNORECASE)

    question = q_match.group(1).strip() if q_match else "Question not generated"
    answer = a_match.group(1).strip() if a_match else "Answer not generated"

    return {
        "flashcard": {
            "question": question,
            "answer": answer
        }
    }

# =========================
# 4. PRESENT QUESTION
# =========================
def present_question(state: BotState):
    print("\n🧠 Question:")
    print(state["flashcard"]["question"])

    ans = input("\n✍️ Your Answer: ")

    return {"learner_answer": ans}

# =========================
# 5. EVALUATE ANSWER (LLM)
# =========================
def evaluate_answer(state: BotState):
    api_key = get_api_key()

    correct_answer = state["flashcard"]["answer"]
    user_answer = state["learner_answer"]

    prompt = f"""
    Compare answers:

    Correct Answer: {correct_answer}
    Student Answer: {user_answer}

    Reply ONLY with:
    CORRECT or INCORRECT
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0
        }
    )

    result = response.json()["choices"][0]["message"]["content"].strip().upper()

    print("🔍 LLM Evaluation:", result)

    # ✅ FIXED LOGIC
    if "INCORRECT" in result:
        is_correct = False
    elif "CORRECT" in result:
        is_correct = True
    else:
        is_correct = False

    if is_correct:
        print("✅ Correct!")
    else:
        print(f"❌ Incorrect! Correct answer: {correct_answer}")

    return {"is_correct": is_correct}

# =========================
# 6. UPDATE PROGRESS
# =========================
def update_progress(state: BotState):
    progress = state.get("progress", [])

    progress.append({
        "question": state["flashcard"]["question"],
        "correct_answer": state["flashcard"]["answer"],
        "your_answer": state["learner_answer"],
        "result": "correct" if state["is_correct"] else "incorrect"
    })

    print("\n📊 Progress:")
    for i, p in enumerate(progress, 1):
        print(f"{i}. {p}")

    return {"progress": progress}

# =========================
# 7. BUILD GRAPH
# =========================
graph = StateGraph(BotState)

graph.add_node("input", input_node)
graph.add_node("generate", generate_flashcard)
graph.add_node("present", present_question)
graph.add_node("evaluate", evaluate_answer)
graph.add_node("history", update_progress)

graph.set_entry_point("input")

graph.add_conditional_edges("input", route_after_input)
graph.add_edge("generate", "present")
graph.add_edge("present", "evaluate")
graph.add_edge("evaluate", "history")
graph.add_edge("history", "input")  # 🔁 loop

app = graph.compile()

# =========================
# 8. RUN
# =========================
if __name__ == "__main__":
    state = {
        "topic": "",
        "flashcard": {},
        "learner_answer": "",
        "is_correct": False,
        "progress": []
    }

    print("🎓 AI Study Assistant (Flashcards)\n")

    app.invoke(state)

🎓 AI Study Assistant (Flashcards)


📘 Enter topic (or 'exit'): exit
👋 Exiting Study Assistant...


# Scenario: Customer Support Call Center
 A company runs a support chatbot that needs to route customer queries to the right department. Instead of one big script, they design a state graph where each node represents a specialized agent.

 1. State Definition (SupportState)
 The chatbot keeps track of:
 - query → What the customer asked.
 - category → Which department it belongs to (billing, tech, general).
 - response → What the bot replies with.
 Think of this as the customer’s “ticket form.”

 2. Router Node (route_query)
 When a customer types a question, the router scans it:
 - If the query mentions “bill” or “payment”, it routes to billing_agent.
 - If it mentions “error” or “bug”, it routes to tech_agent.
 - Otherwise, it defaults to general_agent.
 This is like a receptionist deciding which desk you should go to.

 3. Agent Nodes
 - billing_agent → Replies with “Billing dept: [query]”.
 - tech_agent → Replies with “Tech support: [query]”.
 - general_agent → Replies with “General help: [query]”.
 Each agent specializes in its own type of problem.

 4. Graph Flow
 - Entry point: router.
 - Router decides the path based on the query.
 - The query flows into the correct agent node.
 - The agent generates a response and ends the conversation.

In [7]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class SupportState(TypedDict):
    query: str
    category: str   # "billing" | "tech" | "general"
    response: str

# Router: reads state, returns next node name
def route_query(state: SupportState) -> str:
    q = state["query"].lower()
    if "bill" in q or "payment" in q:
        return "billing_agent"
    elif "error" in q or "bug" in q:
        return "tech_agent"
    else:
        return "general_agent"

def billing_agent(state):
    return {"response": "Billing dept: " + state["query"]}

def tech_agent(state):
    return {"response": "Tech support: " + state["query"]}

def general_agent(state):
    return {"response": "General help: " + state["query"]}

# Build graph with conditional routing
g = StateGraph(SupportState)
g.add_node("billing_agent", billing_agent)
g.add_node("tech_agent", tech_agent)
g.add_node("general_agent", general_agent)

# One entry point routes to 3 different nodes!
g.set_entry_point("router")
g.add_conditional_edges(
    "router",    # from node
    route_query, # function that returns next node
    {            # mapping: return value → node name
        "billing_agent":  "billing_agent",
        "tech_agent":     "tech_agent",
        "general_agent":  "general_agent",
    }
)


#Scenario: AI Symptom Tracker (Question-Based)
1. State Definition
The assistant maintains a notebook-like state for each patient:
- symptom → The patient’s reported issue (e.g., "persistent cough").
- observations → Notes or snippets generated by Groq about possible causes or related conditions.
- analysis → A synthesized interpretation of the observations.
- recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
- steps_done → A counter tracking progress through the workflow.

2. Workflow (Graph of States)
Each patient query flows through nodes:
- Symptom Input Node
- Patient reports a symptom.
- State updates: symptom = "persistent cough"
- Observation Node (Groq API)
- Groq generates possible related factors or general information.
- Updates observations.
- Analysis Node
- Joins observations and extracts key insights.
- Updates analysis.
- Conditional Node
- If fewer than 3 observations are collected → loop back to Observation Node.
- If 3+ observations are available → move to Recommendation Node.
- Recommendation Node
- Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
- Updates recommendation.
- End Node
- Delivers the final recommendation to the patient.

In [8]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import requests
from google.colab import userdata




def get_api_key():
    return userdata.get('NewApi')

# =========================
# 1. STATE
# =========================
class BotState(TypedDict):
    symptom: str
    observations: List[str]
    analysis: str
    recommendation: str
    steps_done: int

# =========================
# 2. INPUT NODE
# =========================
def input_node(state: BotState):
    symptom = input("\n🩺 Enter your symptom (or 'exit'): ")

    if symptom.lower() == "exit":
        print("👋 Exiting Symptom Tracker...")
        return END

    return {
        "symptom": symptom,
        "observations": [],
        "steps_done": 0
    }

# =========================
# 3. OBSERVATION NODE (LLM)
# =========================
def observation_node(state: BotState):
    api_key = get_api_key()

    symptom = state["symptom"]
    observations = state.get("observations", [])

    prompt = f"""
    Provide ONE possible general observation or related factor for the symptom: {symptom}.
    Keep it simple and non-medical advice.
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.7
        }
    )

    if response.status_code != 200:
        raise Exception(response.text)

    obs = response.json()["choices"][0]["message"]["content"].strip()

    observations.append(obs)

    print(f"\n🔎 Observation {len(observations)}: {obs}")

    return {
        "observations": observations,
        "steps_done": state["steps_done"] + 1
    }

# =========================
# 4. ANALYSIS NODE
# =========================
def analysis_node(state: BotState):
    observations = state["observations"]

    analysis = " | ".join(observations)

    print("\n🧠 Analysis:")
    print(analysis)

    return {"analysis": analysis}

# =========================
# 5. CONDITIONAL ROUTER
# =========================
def check_observations(state: BotState):
    if len(state["observations"]) < 3:
        return "observation"
    return "recommendation"

# =========================
# 6. RECOMMENDATION NODE (LLM)
# =========================
def recommendation_node(state: BotState):
    api_key = get_api_key()

    analysis = state["analysis"]

    prompt = f"""
    Based on the following observations:
    {analysis}

    Give a simple, safe, non-medical recommendation.
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.5
        }
    )

    if response.status_code != 200:
        raise Exception(response.text)

    recommendation = response.json()["choices"][0]["message"]["content"]

    print("\n💡 Recommendation:")
    print(recommendation)

    return {"recommendation": recommendation}

# =========================
# 7. BUILD GRAPH
# =========================
graph = StateGraph(BotState)

graph.add_node("input", input_node)
graph.add_node("observation", observation_node)
graph.add_node("analysis", analysis_node)
graph.add_node("recommendation", recommendation_node)

graph.set_entry_point("input")

graph.add_edge("input", "observation")
graph.add_edge("observation", "analysis")

# 🔥 Conditional loop
graph.add_conditional_edges("analysis", check_observations)

graph.add_edge("recommendation", END)

app = graph.compile()

# =========================
# 8. RUN
# =========================
if __name__ == "__main__":
    state = {
        "symptom": "",
        "observations": [],
        "analysis": "",
        "recommendation": "",
        "steps_done": 0
    }

    print("🩺 AI Symptom Tracker\n")

    app.invoke(state)

🩺 AI Symptom Tracker


🩺 Enter your symptom (or 'exit'): persistant cough

🔎 Observation 1: One possible general observation or related factor for a persistent cough is:

Dry air or pollution in the environment.

🧠 Analysis:
One possible general observation or related factor for a persistent cough is:

Dry air or pollution in the environment.

🔎 Observation 2: One possible general observation or related factor for a persistent cough is:

Inadequate hydration or exposure to dry air.

🧠 Analysis:
One possible general observation or related factor for a persistent cough is:

Dry air or pollution in the environment. | One possible general observation or related factor for a persistent cough is:

Inadequate hydration or exposure to dry air.

🔎 Observation 3: One possible general observation or related factor for a persistent cough is environmental pollution or exposure to pollutants such as dust, smoke, or chemicals in the air.

🧠 Analysis:
One possible general observation or related factor

# Scenario: AI-Assisted Email Workflow (Question-Based) Context
 A company deploys an AI-powered email assistant to help employees draft, review, and send professional emails.
 The workflow is modeled as a graph of states, where each email task flows through nodes until it is either approved
 and sent or rejected.

 1. State Definition
 The assistant maintains a notebook-like state:
 - task → The subject or purpose of the email (e.g., "Q3 Report").
 - draft → The AI-generated email draft.
 - approved → A flag indicating whether the human reviewer has approved the draft.

 2. Workflow (Graph of States)
 Each email task flows through nodes:
 - Draft Node
 - AI generates a draft email based on the task.
 - Updates draft.
 - Review Node (Interrupt)
 - Execution pauses here.
 - Human reviewer inspects the draft and decides whether to approve or reject.
 - Updates approved.
 - Send Node
 - If approved = True → Email is sent.
 - If approved = False → Email is rejected.
 - Updates task with final status (SENT or REJECTED).
 - End Node
 - Workflow completes.

3. Example Flow
 - Employee: "Draft an email for the Q3 Report."
 - State: task = "Q3 Report"
 - Assistant drafts:
 Dear User,
 Regarding: Q3 Report
 [AI drafted content]
 - Human reviews → Approves draft (approved = True)
 - Assistant sends → task = "SENT: Q3 Report"
 - Final Output: ✅ Email delivered.

# 👉 Scenario Question:
 "Imagine you are designing an AI-powered email assistant that drafts emails, pauses for human review, and
 then either sends or rejects them. How would you structure the state and workflow graph to ensure accountability
  and human oversight in the process?"

In [9]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class EmailState(TypedDict):
    task: str
    draft: str
    approved: bool

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):
    groq_api_key = userdata.get('NewApi')
    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def draft_email(state: EmailState):
    print(f"📝 Drafting email for task: {state['task']}")
    draft = groq_call(f"Draft a professional email regarding: {state['task']}")
    return {"draft": draft}

def human_review(state: EmailState):
    # Interrupt node: waits for human approval
    print(f"📧 Draft ready for review:\n\n{state['draft']}\n")
    return {}  # Pauses here until human updates 'approved'

def send_email(state: EmailState):
    if state.get("approved", False):
        print("✅ Email approved and sent.")
        return {"task": f"SENT: {state['task']}"}
    else:
        print("❌ Email rejected.")
        return {"task": f"REJECTED: {state['task']}"}

# 4. Build Graph
g = StateGraph(EmailState)
g.add_node("draft", draft_email)
g.add_node("review", human_review)
g.add_node("send", send_email)

g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "send")
g.add_edge("send", END)

# 5. Checkpointer
checkpointer = MemorySaver()
app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]  # Pause before review
)

# 6. Run Workflow
thread = {"configurable": {"thread_id": "email-1"}}

# Step 1: Draft email
app.invoke({"task": "Q3 Report", "draft": "", "approved": False}, thread)

# Step 2: Human reviews draft and resumes
app.invoke({"approved": True}, thread)

📝 Drafting email for task: Q3 Report
📝 Drafting email for task: Q3 Report


{'task': 'Q3 Report',
 'draft': "Subject: Q3 Report and Performance Review\n\nDear [Recipient's Name],\n\nI am writing to inform you that the Q3 report is now available for your review and analysis. As per our quarterly reporting schedule, the document provides a comprehensive overview of our company's performance from July to September.\n\nKey highlights from the Q3 report include:\n\n- Revenue growth of [insert percentage] YoY, meeting our projected targets\n- Net income increase of [insert percentage] compared to Q3 of the previous year\n- Improved customer engagement metrics, including [insert key metrics]\n- Successful implementation of [insert strategic initiatives] to drive business growth\n\nBelow are the key performance indicators (KPIs) for Q3:\n\n- Sales: [insert sales figure]\n- Expenses: [insert expenses figure]\n- Profit Margin: [insert profit margin figure]\n\nIn addition to reviewing the Q3 report, I recommend that we discuss the following initiatives to further drive b

#Scenario Question
"Imagine you are designing an AI-powered assistant that drafts structured reports, pauses for human review, and then either publishes or rejects them. How would you structure the state and workflow graph to ensure accountability and human oversight in the process?"

In [12]:

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from google.colab import userdata
api_key=userdata.get('NewApi')
# ==========================
# GROQ SETUP
# ==========================
llm = ChatGroq(
    groq_api_key=api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0.7
)

# ==========================
# STATE
# ==========================
class ReportState(TypedDict):
    topic: str
    draft: str
    review_status: str
    reviewer_comments: str
    final_output: str

# ==========================
# NODES
# ==========================

# 1. Input Node
def input_topic(state: ReportState):
    topic = input("\n📝 Enter report topic: ")
    return {"topic": topic}

# 2. Draft Generation (AI)
def generate_draft(state: ReportState):
    prompt = f"""
    Create a structured report on topic: {state['topic']}
    Include:
    - Introduction
    - Key Points
    - Conclusion
    """

    draft = llm.invoke(prompt).content

    print("\n📄 Generated Draft:\n")
    print(draft)

    return {"draft": draft, "review_status": "pending"}

# 3. Human Review (HITL)
def human_review(state: ReportState):
    print("\n👨‍💼 Human Review Required")
    print("1. Approve")
    print("2. Reject")

    choice = input("Enter choice (1/2): ")

    if choice == "1":
        return {
            "review_status": "approved",
            "reviewer_comments": "Approved by human"
        }
    else:
        comments = input("Enter feedback for improvement: ")
        return {
            "review_status": "rejected",
            "reviewer_comments": comments
        }

# 4. Decision Node
def review_decision(state: ReportState):
    return state["review_status"]

# 5. Publish Node
def publish(state: ReportState):
    print("\n✅ Report Approved & Published")

    return {
        "final_output": state["draft"]
    }

# 6. Regenerate Node (if rejected)
def regenerate(state: ReportState):
    prompt = f"""
    Improve the report based on feedback:
    Feedback: {state['reviewer_comments']}

    Previous Draft:
    {state['draft']}
    """

    new_draft = llm.invoke(prompt).content

    print("\n🔄 Regenerated Draft:\n")
    print(new_draft)

    return {
        "draft": new_draft,
        "review_status": "pending"
    }

# ==========================
# BUILD GRAPH
# ==========================
builder = StateGraph(ReportState)

builder.add_node("input", input_topic)
builder.add_node("draft", generate_draft)
builder.add_node("review", human_review)
builder.add_node("publish", publish)
builder.add_node("regenerate", regenerate)

# Entry
builder.set_entry_point("input")

# Flow
builder.add_edge("input", "draft")
builder.add_edge("draft", "review")

# Conditional (HITL decision)
builder.add_conditional_edges(
    "review",
    review_decision,
    {
        "approved": "publish",
        "rejected": "regenerate"
    }
)

# Loop back after regeneration
builder.add_edge("regenerate", "review")

builder.add_edge("publish", END)

# Compile
graph = builder.compile()

# ==========================
# RUN
# ==========================
graph.invoke({})


📝 Enter report topic: Ai in tech department

📄 Generated Draft:

**Report: AI in Tech Department**

**Introduction**

The integration of Artificial Intelligence (AI) in the tech department has revolutionized the way organizations operate, manage, and maintain their technological infrastructure. AI technology has enabled businesses to automate routine tasks, enhance decision-making processes, and improve customer experience. This report aims to provide an overview of the key points related to AI in the tech department, highlighting its benefits, applications, and challenges.

**Key Points**

1. **Automation and Efficiency**

- AI has enabled organizations to automate routine tasks, freeing up human resources for more strategic and creative work.
- AI-powered tools have improved the efficiency of tech support, reducing response times and increasing customer satisfaction.

2. **Predictive Maintenance**

- AI has enabled predictive maintenance, allowing organizations to identify potential

{'topic': 'Ai in tech department',
 'draft': '**Report: AI in Tech Department**\n\n**Introduction**\n\nThe integration of Artificial Intelligence (AI) in the tech department has revolutionized the way organizations operate, manage, and maintain their technological infrastructure. AI technology has enabled businesses to automate routine tasks, enhance decision-making processes, and improve customer experience. This report aims to provide an overview of the key points related to AI in the tech department, highlighting its benefits, applications, and challenges.\n\n**Key Points**\n\n1. **Automation and Efficiency**\n\n- AI has enabled organizations to automate routine tasks, freeing up human resources for more strategic and creative work.\n- AI-powered tools have improved the efficiency of tech support, reducing response times and increasing customer satisfaction.\n\n2. **Predictive Maintenance**\n\n- AI has enabled predictive maintenance, allowing organizations to identify potential issu